# Stage 1b — UMLS-Grounded NER (LOCAL / Windows + GPU)

Runs scispaCy biomedical NER + the full UMLS entity linker on your own machine,
so you are NOT limited by Colab RAM. Links each entity to a real UMLS CUI.

**Before you run — read the SETUP GUIDE cell right below.** The first run
downloads ~2 GB (model + UMLS index). With 16 GB RAM, close other heavy apps
(browsers, etc.) while this runs.


## ⚙️ SETUP GUIDE — do this ONCE in a terminal (Anaconda Prompt or CMD)

**1. Make a clean environment (strongly recommended):**
```
conda create -n phenoprompt python=3.10 -y
conda activate phenoprompt
```
(If you don't use conda: `python -m venv phenoprompt` then
`phenoprompt\Scripts\activate`)

**2. Install packages (order matters — pin numpy<2):**
```
pip install "numpy<2"
pip install scispacy==0.6.2 negspacy pandas scikit-learn scipy jupyter
pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz
```

**3. (Optional GPU) If you want the model on GPU:**
```
pip install spacy[cuda12x]
```
Note: the GPU only speeds up the NER model. The UMLS linker uses system RAM
regardless — the GPU does not remove the RAM requirement.

**4. Put your data where this notebook can find it.**
Copy your rule-based Stage 1 output `notes.csv` into a folder, e.g.
`C:\phenoprompt\stage1_outputs\notes.csv`
(That's the file with columns `idx` and `note` you already generated.)

**5. Launch Jupyter from the same terminal:**
```
jupyter notebook
```
Then open this notebook and run the cells top to bottom.

**If anything about numpy/binary-incompatibility appears:** close Jupyter, run
`pip install --force-reinstall "numpy<2" "scipy<1.13"`, then relaunch.


In [6]:
# ── Imports & config (LOCAL paths) ────────────────────────────────────────────
from pathlib import Path
import numpy as np, pandas as pd, time
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import TfidfTransformer
import spacy
from scispacy.abbreviation import AbbreviationDetector
from scispacy.linking import EntityLinker

# >>> EDIT THIS to where you put your data <<<
BASE = Path(r"C:\Users\Dell\Desktop\Phenoprompt")          # your project folder
STAGE1_DIR = BASE / "stage1_outputs"           # must contain notes.csv
OUTPUT_DIR = BASE / "stage1b_outputs"          # results written here
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# With 16 GB RAM you can attempt the FULL run. If it struggles, set a number
# (e.g. 5000) to subsample. None = full corpus.
SUBSAMPLE   = None
RANDOM_SEED = 42

# UMLS semantic types to keep (disorders, findings, procedures, drugs)
KEEP_TUIS = {"T047","T048","T184","T060","T061","T121","T200","T037","T046","T191"}
LINK_THRESHOLD = 0.80
print("Config OK. BASE =", BASE)


Config OK. BASE = C:\Users\Dell\Desktop\Phenoprompt


In [7]:
# ── Load notes ────────────────────────────────────────────────────────────────
notes = pd.read_csv(STAGE1_DIR / "notes.csv").dropna(subset=["note"]).reset_index(drop=True)
notes["idx"] = notes["idx"].astype(str)
print(f"Loaded {len(notes):,} notes.")
if SUBSAMPLE is not None and len(notes) > SUBSAMPLE:
    notes = notes.sample(n=SUBSAMPLE, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Subsampled to {len(notes):,} notes.")


Loaded 30,000 notes.


In [8]:
# ── (Optional) try to use GPU for the model ───────────────────────────────────
try:
    spacy.require_gpu()
    print("GPU enabled for spaCy.")
except Exception as e:
    print("Running on CPU (GPU not enabled):", e)


Running on CPU (GPU not enabled): Cannot use GPU, CuPy is not installed


In [9]:
# ── Build the scispaCy pipeline (first run downloads ~2 GB UMLS index) ─────────
nlp = spacy.load("en_core_sci_md")
nlp.add_pipe("abbreviation_detector")
nlp.add_pipe("scispacy_linker",
             config={"resolve_abbreviations": True,
                     "linker_name": "umls",
                     "threshold": LINK_THRESHOLD,
                     "max_entities_per_mention": 1})
from negspacy.negation import Negex
nlp.add_pipe("negex", config={"ent_types": ["ENTITY"]})
linker = nlp.get_pipe("scispacy_linker")
print("pipeline:", nlp.pipe_names)
print("UMLS linker loaded.")


C:\Users\Dell\miniconda3\envs\phenoprompt\lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/tfidf_vectors_sparse.npz not found in cache, downloading to C:\Users\Dell\AppData\Local\Temp\tmpl7yd3e1l


100%|██████████| 492M/492M [01:23<00:00, 6.19MiB/s]   


Finished download, copying C:\Users\Dell\AppData\Local\Temp\tmpl7yd3e1l to cache at C:\Users\Dell\.scispacy\datasets\2b79923846fb52e62d686f2db846392575c8eb5b732d9d26cd3ca9378c622d40.87bd52d0f0ee055c1e455ef54ba45149d188552f07991b765da256a1b512ca0b.tfidf_vectors_sparse.npz
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/nmslib_index.bin not found in cache, downloading to C:\Users\Dell\AppData\Local\Temp\tmpnuh5ijw8


100%|██████████| 724M/724M [01:47<00:00, 7.07MiB/s]   


...

Finished download, copying C:\Users\Dell\AppData\Local\Temp\tmpo1kpa_vs to cache at C:\Users\Dell\.scispacy\datasets\6238f505f56aca33290aab44097f67dd1b88880e3be6d6dcce65e56e9255b7d4.d7f77b1629001b40f1b1bc951f3a890ff2d516fb8fbae3111b236b31b33d6dcf.concept_aliases.json
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/kbs/2023-04-23/umls_2022_ab_cat0129.jsonl not found in cache, downloading to C:\Users\Dell\AppData\Local\Temp\tmpnr0t93k9


100%|██████████| 628M/628M [01:31<00:00, 7.18MiB/s]   


Finished download, copying C:\Users\Dell\AppData\Local\Temp\tmpnr0t93k9 to cache at C:\Users\Dell\.scispacy\datasets\d5e593bc2d8adeee7754be423cd64f5d331ebf26272074a2575616be55697632.0660f30a60ad00fffd8bbf084a18eb3f462fd192ac5563bf50940fc32a850a3c.umls_2022_ab_cat0129.jsonl
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/umls_semantic_type_tree.tsv not found in cache, downloading to C:\Users\Dell\AppData\Local\Temp\tmpbewq6gab


100%|██████████| 4.26k/4.26k [00:00<?, ?iB/s]

Finished download, copying C:\Users\Dell\AppData\Local\Temp\tmpbewq6gab to cache at C:\Users\Dell\.scispacy\datasets\21a1012c532c3a431d60895c509f5b4d45b0f8966c4178b892190a302b21836f.330707f4efe774134872b9f77f0e3208c1d30f50800b3b39a6b8ec21d9adf1b7.umls_semantic_type_tree.tsv
pipeline: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner', 'abbreviation_detector', 'scispacy_linker', 'negex']
UMLS linker loaded.


In [10]:
# ── Self-test on one sentence (CHECK THIS before the full run) ─────────────────
demo = nlp("Patient has dyspnea and type 2 diabetes. No evidence of pneumonia. "
           "Underwent colonoscopy; started on metformin.")
print(f"{'text':<22}{'CUI':<10}{'canonical name':<32}{'neg?'}")
for e in demo.ents:
    if e._.kb_ents:
        cui, score = e._.kb_ents[0]
        name = linker.kb.cui_to_entity[cui].canonical_name
        print(f"{e.text:<22}{cui:<10}{name[:30]:<32}{getattr(e._,'negex',False)}")
print("\nIf CUIs + names appear (and pneumonia shows neg?=True), it works.")


text                  CUI       canonical name                  neg?
Patient               C0030705  Patients                        False
dyspnea               C0013404  Dyspnea                         False
type 2 diabetes       C0011860  Diabetes Mellitus, Non-Insulin  False
pneumonia             C0032285  Pneumonia                       True
colonoscopy           C0009378  colonoscopy                     False
metformin             C0025598  metformin                       False

If CUIs + names appear (and pneumonia shows neg?=True), it works.


C:\Users\Dell\miniconda3\envs\phenoprompt\lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)


In [11]:
# ── Process ALL notes ─────────────────────────────────────────────────────────
records = []; t0 = time.time()
texts = notes["note"].tolist(); ids = notes["idx"].tolist()
for i, doc in enumerate(nlp.pipe(texts, batch_size=16)):
    nid = ids[i]
    for e in doc.ents:
        if not e._.kb_ents: continue
        cui, score = e._.kb_ents[0]
        ent = linker.kb.cui_to_entity[cui]
        if not (KEEP_TUIS & set(ent.types)): continue
        records.append({
            "note_id": nid, "cui": cui, "concept": ent.canonical_name,
            "text": e.text.lower().strip(), "score": round(float(score),3),
            "assertion": "negated" if bool(getattr(e._,"negex",False)) else "affirmed",
        })
    if (i+1) % 500 == 0:
        print(f"  {i+1:,}/{len(texts):,} notes  ({time.time()-t0:.0f}s)")
ents_df = pd.DataFrame(records)
print(f"\nDone. {len(ents_df):,} linked mentions in {time.time()-t0:.0f}s")
print("assertion counts:", ents_df["assertion"].value_counts().to_dict())
print("distinct CUIs:", ents_df["cui"].nunique())


  500/30,000 notes  (68s)
  1,000/30,000 notes  (138s)
  1,500/30,000 notes  (220s)
  2,000/30,000 notes  (287s)
  2,500/30,000 notes  (354s)
  3,000/30,000 notes  (418s)
  3,500/30,000 notes  (488s)
  4,000/30,000 notes  (552s)
  4,500/30,000 notes  (617s)
  5,000/30,000 notes  (684s)
  5,500/30,000 notes  (752s)
  6,000/30,000 notes  (815s)
  6,500/30,000 notes  (880s)
  7,000/30,000 notes  (945s)
  7,500/30,000 notes  (1011s)
  8,000/30,000 notes  (1079s)
  8,500/30,000 notes  (1145s)
  9,000/30,000 notes  (1210s)
  9,500/30,000 notes  (1274s)
  10,000/30,000 notes  (1344s)
  10,500/30,000 notes  (1409s)
  11,000/30,000 notes  (1474s)
  11,500/30,000 notes  (1538s)
  12,000/30,000 notes  (1605s)
  12,500/30,000 notes  (1672s)
  13,000/30,000 notes  (1738s)
  13,500/30,000 notes  (1802s)
  14,000/30,000 notes  (1870s)
  14,500/30,000 notes  (1935s)
  15,000/30,000 notes  (1999s)
  15,500/30,000 notes  (2064s)
  16,000/30,000 notes  (2133s)
  16,500/30,000 notes  (2197s)
  17,000/30,0

In [12]:
# ── Build affirmed CUI feature matrix + save ──────────────────────────────────
affirmed = ents_df[ents_df["assertion"]=="affirmed"].copy()
cui_doc_freq = affirmed.groupby("cui")["note_id"].nunique()
keep_cuis = cui_doc_freq[cui_doc_freq >= 5].index
affirmed = affirmed[affirmed["cui"].isin(keep_cuis)]
print(f"Concepts kept (CUIs in >=5 notes): {affirmed['cui'].nunique():,}")

count_mat = (affirmed.groupby(["note_id","cui"]).size()
             .unstack(fill_value=0).reindex(ids, fill_value=0))
tfidf   = TfidfTransformer(norm="l2", use_idf=True, smooth_idf=True)
X_tfidf = tfidf.fit_transform(count_mat.values)

count_mat.to_csv(OUTPUT_DIR / "entity_count_matrix.csv")
save_npz(str(OUTPUT_DIR / "entity_tfidf_matrix.npz"), X_tfidf)
np.save(str(OUTPUT_DIR / "entity_vocab.npy"), np.array(count_mat.columns.tolist()))
np.save(str(OUTPUT_DIR / "note_ids.npy"),     np.array(count_mat.index.tolist()))
affirmed[["cui","concept"]].drop_duplicates().to_csv(OUTPUT_DIR / "cui_names.csv", index=False)

print(f"Matrix: {count_mat.shape[0]:,} notes x {count_mat.shape[1]:,} UMLS concepts")
print(f"Saved to {OUTPUT_DIR}")
print("\nTop 15 concepts by document frequency:")
print(affirmed.groupby(['cui','concept'])['note_id'].nunique().sort_values(ascending=False).head(15).to_string())


Concepts kept (CUIs in >=5 notes): 6,963
Matrix: 30,000 notes x 6,963 UMLS concepts
Saved to C:\Users\Dell\Desktop\Phenoprompt\stage1b_outputs

Top 15 concepts by document frequency:
cui       concept                            
C0011900  Diagnosis                              11946
C0040405  X-Ray Computed Tomography               9078
C0024485  Magnetic Resonance Imaging              6672
C0030193  Pain                                    6103
C0015252  removal technique                       5388
C0013604  Edema                                   4241
C0027651  Neoplasms                               4151
C0543467  Operative Surgical Procedures           4015
C0332293  Treated with                            3663
C0041618  Ultrasonography                         3203
C0020538  Hypertensive disease                    3101
C0392148  Providing presence (regime/therapy)     3000
C0184661  Interventional procedure                2718
C0005558  Biopsy                                  2711
C

## Next
Point your Stage 2 notebook at `stage1b_outputs/` (same file names as
`stage1_outputs/`) and re-run to cluster on the UMLS-grounded concepts.
Compare cluster count / silhouette / DBCV / noise against the rule-based run.
